# Проект 1. Что удерживает подписчиков ПРАЙМ: шеринг или канал привлечения?

**Автор:** Артемьев Алексей, БАЦРФ261

## Бизнес-вопрос

Владелец подписки ПРАЙМ может подключить к ней близких. Если подписка, которой пользуется
несколько человек, отменяется реже, ПРАЙМу выгодно продвигать функцию «поделиться»:
каждая удержанная подписка — это продолжение ежемесячных платежей.

**Вопрос: стоит ли ПРАЙМу продвигать шеринг, чтобы снизить отток, — и если нет, где на самом
деле рычаг удержания?**

## Гипотезы

- **H1.** Подписки с участниками уходят реже — и это не объясняется тарифом или возрастом подписки.
- **H2.** Чем больше участников, тем ниже отток.
- **H3.** Общий отток ПРАЙМа меняется за счёт доли подписок с шерингом (структура),
  а не за счёт поведения внутри групп.
- **H4.** Отток определяется каналом, из которого пришёл владелец подписки: клиенты разных
  каналов удерживаются по-разному, и это не объясняется возрастом подписки.

## Договорённости расчёта

- **Зерно — одна строка = подписка × месяц**, только подписки, активные на начало месяца.
  Отток считается именно так (Л5, слайды 25–26): знаменатель — живые на начало месяца,
  числитель — те из них, кто закончился до конца месяца.
- **«С шерингом» — по состоянию на начало месяца**: участник подключился до 1-го числа и ещё
  не вышел. Если брать признак «когда-нибудь делился», получится ошибка в обратную сторону:
  подписка, прожившая долго, успела набрать участников *потому что* долго жила.
- Период — **сентябрь 2025 … июль 2026**, сутки и месяцы по UTC. Август 2026 не берём:
  22–23.08 разом отменены 27 тыс. подписок partner_telecom — это пакетная операция партнёра,
  а не решения клиентов, и она исказила бы отток.
- Выручку считаем **только по `payments`**: участников к платежам не присоединяем, иначе
  платёж повторится на каждого участника (двойной счёт из-за шеринга).

## 1. Подключение и параметры

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine
from prime.factors import contrib, waterfall
from prime.metrics import churn_rate

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))      # сутки и месяцы — по UTC
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


FIRST, LAST = "2025-09-01", "2026-07-01"   # начала первого и последнего месяца периода
M0, M1 = "2025-10", "2026-07"              # что с чем сравниваем в разложении (оба по 31 дню)
TARIFF = {1: "Базовый", 2: "Расширенный"}
GREY, ACCENT, DARK = "#9CA3AF", "#C1272D", "#374151"
SOURCE = "Источник: prime.subscriptions, prime.subscription_members. Месяцы по UTC."

plt.rcParams["font.size"] = 11
print("подключились как:", q("select current_user as login").iloc[0, 0])

## 2. Данные и их приёмка

Две таблицы: подписки и участники. Колонки берём только нужные — `select *` по участникам
привёз бы лишнее.

In [ ]:
# Канал подписки — это канал её владельца (как на семинаре 5).
subs = q("""
    select s.subscription_id, s.owner_client_id, s.tariff_id, s.started_at, s.ended_at,
           s.is_trial, c.acquisition_channel
    from prime.subscriptions s
    join prime.clients c on c.client_id = s.owner_client_id
""")
# Клиентов без канала не выбрасываем и не прячем в «прочее» — отдельная строка (С3, С6).
print("подписок без канала:", subs["acquisition_channel"].isna().sum())
subs["channel"] = subs["acquisition_channel"].fillna("без канала")
members = q("""
    select subscription_id, client_id, joined_at, left_at
    from prime.subscription_members
""")
tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")

print(f"подписок:  {len(subs):>8}   уникальных id: {subs['subscription_id'].nunique()}")
print(f"участий:   {len(members):>8}")
print(subs.dtypes, members.dtypes, sep="\n\n")
tariffs

In [ ]:
# Приёмка участников: значения, которых не может быть, и противоречия между колонками (С3).
own = members.merge(subs[["subscription_id", "owner_client_id", "tariff_id", "started_at", "ended_at"]],
                    on="subscription_id", how="left", validate="m:1", indicator=True)
slots = tariffs.groupby("tariff_id")["share_slots"].max()

checks = pd.Series({
    "участие без подписки в subscriptions": (own["_merge"] == "left_only").sum(),
    "дубль: тот же человек в той же подписке": members.duplicated(["subscription_id", "client_id"]).sum(),
    "владелец числится участником своей подписки": (own["client_id"] == own["owner_client_id"]).sum(),
    "вышел раньше, чем вошёл": (members["left_at"] < members["joined_at"]).sum(),
    "вошёл раньше, чем подписка началась": (own["joined_at"] < own["started_at"]).sum(),
    "вошёл после того, как подписка закончилась": (own["joined_at"] > own["ended_at"]).sum(),
})
print("мест для участников по тарифам (share_slots):", slots.to_dict())
checks.to_frame("строк")

*Что видно:* таблица почти чистая. Дублей и участий без подписки нет, «вышел раньше, чем вошёл» — ни одного.
Один владелец числится участником собственной подписки — на результат не влияет. **13 421 участие оформлено
уже после того, как подписка закончилась** — это дефект данных, но на признак «с шерингом» он не действует:
такие подписки на начало месяца уже не активны и в таблицу фактов не попадают.

## 3. Таблица фактов: подписка × месяц

Для каждого месяца берём подписки, **активные на его начало**, и для каждой считаем, сколько
участников было подключено **на ту же дату**. Отток — закончилась ли подписка до конца месяца.

In [ ]:
AGE_BINS = [0, 3, 6, 12, np.inf]
AGE_LABELS = ["до 3 мес", "3–6 мес", "6–12 мес", "12+ мес"]

frames = []
for ms in pd.date_range(FIRST, LAST, freq="MS", tz="UTC"):
    me = ms + pd.DateOffset(months=1)
    active = subs[(subs["started_at"] < ms) & (subs["ended_at"].isna() | (subs["ended_at"] >= ms))]
    on_date = members[(members["joined_at"] < ms) & (members["left_at"].isna() | (members["left_at"] >= ms))]
    k = on_date.groupby("subscription_id").size()

    f = active[["subscription_id", "tariff_id", "is_trial", "channel"]].copy()
    f["month"] = ms.strftime("%Y-%m")
    f["n_members"] = f["subscription_id"].map(k).fillna(0).astype("int64")
    f["churned"] = (active["ended_at"] < me).to_numpy()
    f["age_months"] = ((ms - active["started_at"]).dt.days // 30).to_numpy()
    frames.append(f)

panel = pd.concat(frames, ignore_index=True)
panel["shared"] = panel["n_members"] > 0
panel["group"] = np.where(panel["shared"], "с участниками", "без участников")
panel["age"] = pd.cut(panel["age_months"], bins=AGE_BINS, labels=AGE_LABELS, right=False)
panel["tariff"] = panel["tariff_id"].map(TARIFF)

print(len(panel), "строк")
panel.head()

### Протокол: зерно и отток против независимого контроля

Отток по таблице фактов сверяем с функцией `churn_rate` с семинара 5, которая считает его
прямо по `subscriptions`, без участников и без этой таблицы. Если не совпадёт — значит,
таблица фактов собрана неверно.

In [ ]:
months = panel["month"].unique()
grain_ok = not panel.duplicated(["subscription_id", "month"]).any()

panel_churn = panel.groupby("month")["churned"].mean()
control_churn = pd.Series({
    mo: churn_rate(subs, f"{mo}-01", (pd.Period(mo, freq="M") + 1).start_time.strftime("%Y-%m-%d"))
    for mo in months})
churn_diff = (panel_churn - control_churn).abs().max()
missing_age = panel["age"].isna().sum()
over_slots = (panel["n_members"] > panel["tariff_id"].map(slots)).sum()

print(f"ЗЕРНО   дублей (подписка, месяц): {panel.duplicated(['subscription_id', 'month']).sum()}"
      f"   -> {'ok' if grain_ok else 'ВНИМАНИЕ'}")
print(f"ОТТОК   макс. расхождение с churn_rate по месяцам: {churn_diff:.2e}"
      f"   -> {'ok' if churn_diff < 1e-12 else 'ВНИМАНИЕ'}")
print(f"ВОЗРАСТ строк без возрастной группы: {missing_age}"
      f"   -> {'ok' if missing_age == 0 else 'ВНИМАНИЕ'}")
print(f"МЕСТА   участников на начало месяца больше, чем мест в тарифе: {over_slots}"
      f"   -> {'ok' if over_slots == 0 else 'ВНИМАНИЕ — посмотреть глазами'}")
assert grain_ok and churn_diff < 1e-12 and missing_age == 0

pd.DataFrame({"активных на начало": panel.groupby("month").size(),
              "доля с участниками": panel.groupby("month")["shared"].mean().round(4),
              "отток": panel_churn.round(4)})

## 4. H1: уходят ли подписки с участниками реже

In [ ]:
by_month = panel.pivot_table(index="month", columns="group", values="churned", aggfunc="mean")
by_month["разница, п. п."] = (by_month["без участников"] - by_month["с участниками"]) * 100
(by_month * [100, 100, 1]).round(2)      # отток в %, разница — в процентных пунктах

In [ ]:
# ГРАФИК 1. Заголовок — это вывод.
TITLE_1 = "Шеринг не снижает отток: с участниками и без — около 4 % в месяц"

fig, ax = plt.subplots(figsize=(9.5, 4.4), layout="constrained")
for name, color in (("без участников", GREY), ("с участниками", ACCENT)):
    ax.plot(by_month.index, by_month[name] * 100, marker="o", color=color, linewidth=2.2)
    ax.annotate(name, (by_month.index[-1], by_month[name].iloc[-1] * 100), xytext=(8, 0),
                textcoords="offset points", color=color, va="center")
ax.set_title(TITLE_1, loc="left")
ax.set_ylabel("отток за месяц, %")
ax.set_ylim(0, by_month[["без участников", "с участниками"]].max().max() * 100 * 1.2)
ax.tick_params(axis="x", labelrotation=45)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB")
fig.text(0.01, -0.03, SOURCE + " Отток = закончились в месяце / активны на его начало; "
         "«с участниками» — на 1-е число месяца.", fontsize=8, color="#6B7280")
plt.show()

### Проверка: может, дело не в шеринге, а в тарифе или возрасте подписки?

Два подозреваемых:

- **тариф** — у тарифов разное число мест для участников, а отток у тарифов может различаться
  сам по себе;
- **возраст подписки** — старые подписки уходят реже (кто хотел уйти, ушёл раньше) и при этом
  успели набрать участников. Тогда разница H1 — просто разница «старые против новых».

Сравниваем с шерингом и без **внутри** каждого тарифа и каждой возрастной группы. Если разница
держится внутри групп — дело не в них.

In [ ]:
by_tariff = panel.pivot_table(index="tariff", columns="group", values="churned", aggfunc="mean") * 100
by_tariff["подписко-месяцев"] = panel.groupby("tariff").size()
by_tariff["доля с участниками, %"] = panel.groupby("tariff")["shared"].mean() * 100
by_tariff.round(2)

In [ ]:
by_age = panel.pivot_table(index="age", columns="group", values="churned", aggfunc="mean",
                           observed=True) * 100
by_age["доля с участниками, %"] = panel.groupby("age", observed=True)["shared"].mean() * 100
by_age["подписко-месяцев"] = panel.groupby("age", observed=True).size()
by_age.round(2)

In [ ]:
# ГРАФИК 2. Разница внутри возрастных групп.
TITLE_2 = "Внутри каждой возрастной группы разница — не больше 0,1 п. п."

x = np.arange(len(by_age))
fig, ax = plt.subplots(figsize=(9, 4.4), layout="constrained")
b0 = ax.bar(x - 0.2, by_age["без участников"], width=0.4, color=GREY, zorder=2, label="без участников")
b1 = ax.bar(x + 0.2, by_age["с участниками"], width=0.4, color=ACCENT, zorder=2, label="с участниками")
for b in (b0, b1):
    ax.bar_label(b, fmt=lambda v: f"{v:.1f}".replace(".", ","), padding=2, fontsize=9)
ax.set_xticks(x, by_age.index)
ax.set_title(TITLE_2, loc="left")
ax.set_ylabel("отток за месяц, %")
ax.set_xlabel("возраст подписки на начало месяца")
ax.set_ylim(0, by_age[["без участников", "с участниками"]].max().max() * 1.2)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB")
fig.text(0.01, -0.03, SOURCE + " Все месяцы сен 2025 – июл 2026 вместе.", fontsize=8, color="#6B7280")
plt.show()

*Что видно:* **H1 не подтвердилась.** Отток с участниками и без почти одинаков: по месяцам разница скачет
от −0,14 до +0,10 п. п. и меняет знак — это шум, а не эффект. Внутри тарифов (Базовый 3,96 % против 3,94 %,
Расширенный 3,92 % против 3,94 %) и внутри возрастных групп картина та же. Отток зависит от возраста подписки
(до 3 месяцев — 4,2 %, старше года — 3,7 %), но не от шеринга.

## 5. H2: чем больше участников, тем ниже отток?

In [ ]:
by_n = panel.groupby("n_members").agg(sub_months=("churned", "size"), churn=("churned", "mean"))
by_n["churn"] *= 100
by_n.round(2)

*Что видно:* **H2 не подтвердилась.** От 0 до 5 участников отток стоит на месте: 3,95 %, 3,92 %, 3,92 %,
3,96 %, 3,92 %, 4,05 %. Связи «больше участников — меньше уходят» нет.

## 6. H3: факторное разложение изменения оттока, октябрь 2025 → июль 2026

Общий отток — средневзвешенное оттока двух групп:

> отток = Σ по группам ( доля группы × отток группы )

Значит, изменение общего оттока раскладывается на **эффект структуры** (сменилась доля
подписок с участниками) и **эффект поведения** (сменился отток внутри групп) — Л6, слайды 14–15:
сложение по группам, умножение внутри.

Цепные подстановки: сначала «сколько» — доля группы, потом «как» — отток группы.
Сравниваем октябрь 2025 и июль 2026: оба по 31 дню (у месяца в 30 дней отток меньше просто
потому, что в нём меньше дней), оба без дыр и без августовского обвала.

In [ ]:
g = panel[panel["month"].isin([M0, M1])].groupby(["month", "group"]).agg(
    n=("churned", "size"), churn=("churned", "mean"))
g["share"] = g["n"] / g.groupby("month")["n"].transform("sum")

total0 = panel.loc[panel["month"] == M0, "churned"].mean()
total1 = panel.loc[panel["month"] == M1, "churned"].mean()

effects = {}
for grp in ["без участников", "с участниками"]:
    x0 = {"share": g.loc[(M0, grp), "share"], "churn": g.loc[(M0, grp), "churn"]}
    x1 = {"share": g.loc[(M1, grp), "share"], "churn": g.loc[(M1, grp), "churn"]}
    e = contrib(x0, x1, ["share", "churn"])
    effects[f"доля «{grp}»"] = e["share"]
    effects[f"отток «{grp}»"] = e["churn"]

effects = pd.Series(effects) * 100                       # в процентных пунктах
assert abs(effects.sum() - (total1 - total0) * 100) < 1e-9, "сумма влияний не равна изменению оттока"

print(g.assign(share=g["share"] * 100, churn=g["churn"] * 100).round(3))
print(f"\nобщий отток: {total0:.2%} → {total1:.2%}, изменение {(total1 - total0) * 100:+.3f} п. п.")
print(f"  структура (доли групп):     {effects.filter(like='доля').sum():+.3f} п. п.")
print(f"  поведение (отток в группах): {effects.filter(like='отток').sum():+.3f} п. п.")
effects.round(3)

In [ ]:
# Порядок подстановок — часть ответа. Проверяем обратный порядок: сначала отток, потом доля.
rev = {}
for grp in ["без участников", "с участниками"]:
    x0 = {"share": g.loc[(M0, grp), "share"], "churn": g.loc[(M0, grp), "churn"]}
    x1 = {"share": g.loc[(M1, grp), "share"], "churn": g.loc[(M1, grp), "churn"]}
    rev[grp] = contrib(x0, x1, ["churn", "share"])
rev = pd.DataFrame(rev).T * 100
pd.DataFrame({"сначала доля": [effects.filter(like="доля").sum(), effects.filter(like="отток").sum()],
              "сначала отток": [rev["share"].sum(), rev["churn"].sum()]},
             index=["структура", "поведение"]).round(3)

*Что видно:* **H3 не подтвердилась.** Общий отток снизился с 4,06 % до 3,92 % (−0,14 п. п.), но доля подписок
с участниками почти не изменилась (65,3 % → 65,6 %), поэтому эффект структуры — ноль. Всё изменение — поведение
внутри групп, и одинаковое в обеих. Шеринг не объясняет и динамику оттока. Вывод от порядка подстановок
не зависит: в обратном порядке получается то же самое.

## 7. Сколько это стоит в рублях

Переводим разницу в оттоке в деньги на июле 2026:

- **ARPPU** — выручка июля на платящую подписку, по группам. Считаем по `payments` и только
  по подпискам, активным на 1 июля: так знаменатель тот же, что у оттока.
- **LTV ≈ ARPPU / отток** — сколько подписка принесёт за жизнь, если отток и платёж не изменятся
  (Л5, слайд 27). Это прогноз, а не факт.
- **Удержанная выручка** — сколько подписок с участниками ушло бы в июле при оттоке группы
  без участников, умножить на ARPPU.

In [ ]:
pay_jul = q("""
    select subscription_id, count(*) as n_payments, sum(amount) as revenue
    from prime.payments
    where status = 'success' and paid_at >= '2026-07-01' and paid_at < '2026-08-01'
    group by 1
""")

jul = panel[panel["month"] == "2026-07"].merge(pay_jul, on="subscription_id", how="left",
                                               validate="1:1")
jul[["n_payments", "revenue"]] = jul[["n_payments", "revenue"]].fillna(0)

money = jul.groupby("group").agg(subs=("subscription_id", "size"), churn=("churned", "mean"),
                                 paying=("n_payments", lambda s: (s > 0).sum()),
                                 revenue=("revenue", "sum"))
money["arppu"] = money["revenue"] / money["paying"]
money["ltv"] = money["arppu"] / money["churn"]

c_no, c_sh = money.loc["без участников", "churn"], money.loc["с участниками", "churn"]
n_sh = money.loc["с участниками", "subs"]
saved_subs = n_sh * (c_no - c_sh)
saved_rub = saved_subs * money.loc["с участниками", "arppu"]

print(f"подписок с участниками на 1 июля: {n_sh:,}".replace(",", " "))
print(f"при оттоке «без участников» ушло бы на {saved_subs:,.0f} подписок больше".replace(",", " "))
print(f"это ~{saved_rub:,.0f} ₽".replace(",", " ") + " ежемесячной выручки, удержанной за один месяц")
money.round({"churn": 4, "arppu": 2, "ltv": 0})

## 8. H4: отток по каналам привлечения

Шеринг не объяснил разницу в оттоке. Проверяем другой признак — канал, из которого пришёл
владелец подписки. Если каналы удерживают по-разному, это и есть рычаг: решения о бюджете
маркетинга принимаются как раз по каналам.

In [ ]:
by_ch = panel.groupby("channel").agg(sub_months=("churned", "size"), churn=("churned", "mean"),
                                     shared=("shared", "mean"))
by_ch[["churn", "shared"]] *= 100
by_ch = by_ch.sort_values("churn")
avg_churn = panel["churned"].mean() * 100
print(f"средний отток по всем подписко-месяцам: {avg_churn:.2f} %")
by_ch.round(2).rename(columns={"sub_months": "подписко-месяцев", "churn": "отток, %",
                               "shared": "доля с участниками, %"})

In [ ]:
# Проверка: может, дело в возрасте? Новые подписки уходят чаще (раздел 4), и если в каком-то
# канале больше молодых подписок, его отток будет выше «сам собой». Смотрим каналы внутри
# каждой возрастной группы.
ch_age = panel.pivot_table(index="channel", columns="age", values="churned", aggfunc="mean",
                           observed=True) * 100
ch_age.loc[by_ch.index].round(2)

In [ ]:
# LTV по каналам ≈ ARPPU / отток (Л5, слайд 27).
# ARPPU — июль 2026 (подписки, активные на 1 июля). Отток — средний за все 11 месяцев:
# месячный отток канала шумит, а LTV делится на него, и шум раздувается.
ltv_ch = jul.groupby("channel").agg(paying=("n_payments", lambda s: (s > 0).sum()),
                                    revenue=("revenue", "sum"))
ltv_ch["arppu"] = ltv_ch["revenue"] / ltv_ch["paying"]
ltv_ch["churn"] = by_ch["churn"] / 100
ltv_ch["ltv"] = ltv_ch["arppu"] / ltv_ch["churn"]
ltv_ch["life_months"] = 1 / ltv_ch["churn"]
ltv_ch = ltv_ch.loc[by_ch.index]
ltv_ch.round({"arppu": 2, "churn": 4, "ltv": 0, "life_months": 1})

In [ ]:
# ГРАФИК 3. Отток и LTV по каналам.
TITLE_3 = "Отток social втрое выше, чем у organic: 7,0 % против 2,2 % в месяц"

colors = [ACCENT if c > avg_churn else GREY for c in by_ch["churn"]]
fig, ax = plt.subplots(figsize=(9.5, 4.6), layout="constrained")
bars = ax.barh(by_ch.index, by_ch["churn"], color=colors, zorder=2)
ax.bar_label(bars, labels=[f"{c:.2f}".replace(".", ",") + " %  ·  LTV "
                           + f"{l:,.0f}".replace(",", " ") + " ₽"
                           for c, l in zip(by_ch["churn"], ltv_ch["ltv"])], padding=4, fontsize=9)
ax.axvline(avg_churn, color=DARK, linestyle="--", linewidth=1)
ax.annotate(f"в среднем {avg_churn:.2f} %".replace(".", ","), (avg_churn, -0.6),
            xytext=(4, 0), textcoords="offset points", fontsize=9, color=DARK, va="center")
ax.set_title(TITLE_3, loc="left")
ax.set_xlabel("отток за месяц, %, в среднем за сен 2025 – июл 2026")
ax.set_xlim(0, by_ch["churn"].max() * 1.45)
ax.set_ylim(len(by_ch) - 0.5, -1.1)        # сверху — самый низкий отток; место под подпись среднего
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#E5E7EB")
fig.text(0.01, -0.03, "Источник: prime.subscriptions, prime.clients, prime.payments. Канал — владельца "
         "подписки. Красным — каналы с оттоком выше среднего. LTV ≈ ARPPU июля / отток.",
         fontsize=8, color="#6B7280")
plt.show()

### Факторное разложение по каналам: октябрь 2025 → июль 2026

То же разложение, что в разделе 6, но группы — каналы:

> отток = Σ по каналам ( доля канала в активных подписках × отток канала )

**Эффект структуры** — база сместилась к каналам с другим оттоком. **Эффект поведения** —
внутри канала стали уходить чаще или реже. Порядок подстановок прежний: сначала доля, потом отток.

In [ ]:
gc = panel[panel["month"].isin([M0, M1])].groupby(["month", "channel"]).agg(
    n=("churned", "size"), churn=("churned", "mean"))
gc["share"] = gc["n"] / gc.groupby("month")["n"].transform("sum")

eff_ch = {}
for ch in by_ch.index:
    x0 = {"share": gc.loc[(M0, ch), "share"], "churn": gc.loc[(M0, ch), "churn"]}
    x1 = {"share": gc.loc[(M1, ch), "share"], "churn": gc.loc[(M1, ch), "churn"]}
    eff_ch[ch] = contrib(x0, x1, ["share", "churn"])
eff_ch = pd.DataFrame(eff_ch).T * 100                    # процентные пункты
assert abs(eff_ch.values.sum() - (total1 - total0) * 100) < 1e-9, "каналы не сходятся с изменением оттока"

table = gc.unstack("month")
table = pd.DataFrame({
    "доля Окт, %": table[("share", M0)] * 100, "доля Июл, %": table[("share", M1)] * 100,
    "отток Окт, %": table[("churn", M0)] * 100, "отток Июл, %": table[("churn", M1)] * 100,
    "эффект доли, п. п.": eff_ch["share"], "эффект оттока, п. п.": eff_ch["churn"],
}).loc[by_ch.index]
print(f"общий отток {total0:.2%} → {total1:.2%}: изменение {(total1 - total0) * 100:+.3f} п. п.")
print(f"  структура (состав по каналам): {eff_ch['share'].sum():+.3f} п. п.")
print(f"  поведение (отток внутри каналов): {eff_ch['churn'].sum():+.3f} п. п.")
table.round(3)

In [ ]:
# ГРАФИК 4. Каскад изменения оттока по каналам.
TITLE_4 = "Отток снизился на 0,14 п. п.: половина — база сместилась от social и performance"

steps = pd.concat([pd.Series({"структура (состав каналов)": eff_ch["share"].sum()}),
                   eff_ch["churn"]])
ax = waterfall(steps, title=TITLE_4, unit="процентные пункты", decimals=2, lower_is_better=True)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=9)
ax.set_xlabel("Октябрь 2025 → июль 2026, оба по 31 дню. Первый столбик — сдвиг состава подписок по каналам,\n"
              "остальные — изменение оттока внутри канала. Порядок подстановок: доля канала, затем его отток.")
plt.show()

*Что видно:* **H4 подтвердилась.** Каналы различаются в разы: organic теряет 2,2 % подписок в месяц,
social — 7,0 %, performance — 5,4 %. Разница держится внутри каждой возрастной группы (organic 2,2–2,3 %,
social 6,9–7,1 %), значит, дело не в том, что в каком-то канале больше новых подписок. Доля подписок с участниками
во всех каналах одна и та же, около 65 %, — то есть шеринг и здесь ни при чём.

При одинаковом ARPPU (335–337 ₽) LTV определяется только оттоком: подписка из organic приносит ~15,0 тыс. ₽,
из social — ~4,8 тыс. ₽, втрое меньше.

Разложение: из −0,14 п. п. изменения общего оттока **−0,07 дала структура** — доля social упала с 12,4 % до 11,3 %,
performance — с 19,3 % до 18,4 %, organic выросла с 27,7 % до 29,1 %; **−0,06 — поведение** внутри каналов,
и оно разнонаправленное (referral и partner_telecom чуть хуже, organic и crm чуть лучше).

## 9. Вывод

**Что нашли.** С сентября 2025 по июль 2026 подписки ПРАЙМ уходят со скоростью около 4 % в месяц. Подписки,
которыми владелец поделился с близкими (65 % базы), уходят так же часто, как и остальные: разница меньше
0,1 п. п. в любом месяце, тарифе и возрастной группе, и от числа участников отток не зависит. Зато отток
в разы различается по каналу привлечения: organic — 2,2 % в месяц, social — 7,0 %.

**За счёт чего.** Общий отток снизился с 4,06 % (октябрь 2025) до 3,92 % (июль 2026). Разложение по каналам:
−0,07 п. п. — база сместилась от social и performance к organic, −0,06 п. п. — изменение оттока внутри каналов.
По шерингу эффект структуры нулевой: его доля за год не изменилась. При одинаковом ARPPU LTV подписки из organic
(~15 тыс. ₽) втрое выше, чем из social (~4,8 тыс. ₽).

**Чего мы не знаем.** Это наблюдение, а не эксперимент: мы видим связь канала с оттоком, но не знаем, почему
клиенты social уходят быстрее. Не учтены расходы на привлечение (CAC) — без них нельзя сказать, какой канал
окупается. Шеринг может влиять на что-то кроме оттока (покупки в сервисах, привлечение участников в собственные
подписки) — это здесь не проверялось. 13 421 участие оформлено после окончания подписки — источник этого
дефекта надо выяснить у владельцев данных.

**Что делать.** (1) Не тратить бюджет на продвижение шеринга как инструмента удержания. (2) Маркетингу — сопоставить
LTV из этой записки с CAC по каналам и перераспределить бюджет в пользу каналов с низким оттоком (organic, referral,
crm). (3) Продуктовой команде — разобраться, почему уходят клиенты social и performance, и проверить меры удержания
для них пилотом с контрольной группой.